# Lezione 1 — Introduzione a NLP & Generative AI + Setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_1_introduzione/notebook_01_introduzione.ipynb)

Benvenuti al corso **NLP & Generative AI con modelli open source**! 👋

In questa prima lezione:
1. prepariamo l'ambiente (Google Colab + **GPU T4**);
2. capiamo *cosa sono* NLP, LLM e Generative AI;
3. conosciamo l'ecosistema **Hugging Face** + **LangChain**;
4. eseguiamo la nostra **prima pipeline di NLP** su dati reali.

> 🎯 **Filo conduttore del corso:** lavoreremo sempre sullo stesso scenario —
> l'**analisi di recensioni clienti in italiano** — aggiungendo un pezzo per lezione
> fino a costruire, nell'ultima, un'applicazione completa con **RAG**.

---
### ⚙️ Prima di tutto: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
Senza GPU le lezioni successive saranno molto lente.

## 1. Verifichiamo l'ambiente

Controlliamo che PyTorch veda la GPU. Su Colab PyTorch è già installato.

In [ ]:
import torch

print("Versione PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Scheda:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU non attiva. Vai su Runtime > Change runtime type > T4 GPU.")

## 2. Installiamo le librerie

Per questa lezione basta `transformers` (la libreria di Hugging Face per usare i
modelli). `torch` e `pandas` sono già presenti su Colab.

> 💡 Ogni notebook del corso installa da solo ciò che gli serve, così puoi aprire
> qualsiasi lezione in modo indipendente.

In [ ]:
# -q = silenzioso. Su Colab l'installazione richiede qualche secondo.
!pip install -q "transformers>=4.45" "huggingface_hub>=0.25"
print("Librerie installate ✅")

## 3. Il nostro dataset: recensioni clienti 🛒

Useremo un dataset **sintetico** di recensioni in italiano, generato da uno script
del repository (`dati/genera_recensioni.py`). È *riproducibile* (seed fisso) e non
richiede download esterni o gestione di licenze.

La cella seguente scarica lo script (se non già presente) e genera le recensioni.

In [ ]:
import os
import pandas as pd

# Scarica lo script generatore se non è già nella sessione Colab.
if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

recensioni = genera_recensioni.genera_recensioni(n=200, seed=42)
df = pd.DataFrame(recensioni)

print("Numero di recensioni:", len(df))
print("Distribuzione stelle:")
print(df["rating"].value_counts().sort_index(ascending=False))
df.head(3)

In [ ]:
# Diamo un'occhiata a una recensione completa.
esempio = df.iloc[1]
print(f"Prodotto : {esempio['prodotto']}")
print(f"Stelle   : {esempio['rating']} ⭐")
print(f"Titolo   : {esempio['titolo']}")
print(f"Testo    : {esempio['testo']}")

## 4. Un po' di teoria (in pillole) 🧠

- **NLP (Natural Language Processing):** insieme di tecniche per far "capire" e
  manipolare il linguaggio umano ai computer (classificare, estrarre, tradurre…).
- **Modello pre-addestrato:** una rete neurale già allenata su enormi quantità di
  testo. Noi la *usiamo* (e a volte la adattiamo), non la addestriamo da zero.
- **Token:** i modelli non leggono parole intere ma "pezzi" di parola (token).
  👉 Provalo: <https://huggingface.co/spaces/Xenova/the-tokenizer-playground>
- **Transformer:** l'architettura alla base dei modelli moderni.
  👉 Spiegazione visuale: <https://jalammar.github.io/illustrated-transformer/>
- **LLM (Large Language Model):** un Transformer molto grande che *genera* testo
  prevedendo il token successivo (es. la famiglia GPT, Llama, **Qwen**…).
- **Generative AI:** uso degli LLM per *creare* contenuti (risposte, riassunti,
  codice…), non solo per classificare.

**Due famiglie di strumenti che useremo:**
- 🤗 **Hugging Face** — l'"app store" dei modelli open source: li scarichiamo e
  li eseguiamo in locale/Colab, **senza API key**.
- 🦜 **LangChain** — il framework per orchestrare gli LLM in applicazioni
  (prompt, catene, memoria, RAG). Lo introduciamo dalla Lezione 5.

> **Modelli open vs API a pagamento:** con i modelli open di Hugging Face non
> condividiamo dati con servizi esterni e non paghiamo per chiamata. In cambio
> serve hardware (qui: la GPU T4 di Colab).

## 5. La nostra prima pipeline di NLP 🚀

Hugging Face offre le **`pipeline`**: un modo in una riga per usare un modello su
un compito specifico. Iniziamo con l'**analisi del sentiment** usando un modello
addestrato sull'italiano: `neuraly/bert-base-italian-cased-sentiment`, che classifica
un testo come **negativo**, **neutro** o **positivo**.

In [ ]:
from transformers import pipeline

# device=0 -> usa la GPU; -1 -> CPU. (Questo modello è piccolo, va bene comunque.)
device = 0 if torch.cuda.is_available() else -1

classificatore_sentiment = pipeline(
    task="text-classification",
    model="neuraly/bert-base-italian-cased-sentiment",
    device=device,
)

# Il modello restituisce una di tre etichette: 'negative', 'neutral', 'positive'.
# Proviamolo su due frasi.
print(classificatore_sentiment("Servizio pessimo, non lo ricomprerò mai più."))
print(classificatore_sentiment("Prodotto fantastico, arrivato in un giorno!"))

In [ ]:
# Applichiamolo a TUTTE le recensioni e aggiungiamo la colonna 'sentiment_predetto'.
testi = df["testo"].tolist()
predizioni = classificatore_sentiment(testi, batch_size=16, truncation=True)

df["sentiment_predetto"] = [p["label"] for p in predizioni]
df[["rating", "titolo", "sentiment_predetto", "testo"]].head(8)

## 6. Esercizio 🏋️

Il modello ci dà `negative` / `neutral` / `positive`. Le recensioni hanno anche le
**stelle**. Verifichiamo *quanto sono d'accordo*: consideriamo "positiva" una recensione
con **rating ≥ 4** e "negativa" con **rating ≤ 2** (il 3 è ambiguo, lo escludiamo). Per il
confronto teniamo solo le previsioni nette del modello (escludiamo i `neutral`).

Completa il calcolo della percentuale di accordo tra modello e stelle.

In [ ]:
# Teniamo solo le recensioni "nette" (rating != 3).
nette = df[df["rating"] != 3].copy()

# Sentiment "vero" dedotto dalle stelle.
nette["sentiment_da_stelle"] = nette["rating"].apply(
    lambda r: "positive" if r >= 4 else "negative"
)

# Per il confronto teniamo solo le previsioni nette del modello (no 'neutral').
confronto = nette[nette["sentiment_predetto"].isin(["positive", "negative"])]

# TODO: calcola la percentuale di accordo tra 'sentiment_predetto' e
#       'sentiment_da_stelle'. Suggerimento: confronta le due colonne con ==.
accordo = (confronto["sentiment_predetto"] == confronto["sentiment_da_stelle"]).mean()
n_neutral = int((nette["sentiment_predetto"] == "neutral").sum())
print(f"Accordo modello vs stelle: {accordo:.1%}  (su {len(confronto)} recensioni nette)")
print(f"Recensioni 'neutral' (escluse dal confronto): {n_neutral}")

## 7. Riepilogo e prossimi passi ✅

Oggi abbiamo:
- attivato la **GPU T4** e installato `transformers`;
- caricato il dataset di **recensioni** (il nostro filo conduttore);
- capito i concetti base (token, Transformer, LLM, GenAI);
- eseguito la prima **pipeline** di sentiment in **una riga**.

➡️ **Prossima lezione:** come si *rappresenta* il testo in numeri (**embeddings**)
e come fare **ricerca semantica** tra le recensioni — la base del motore di ricerca
del nostro progetto finale.

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026